# **Importing and downloading the needed libraries**

In [ ]:
!pip install numpy==1.25.2

In [ ]:
!pip install gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 58.7 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.15.3
    Uninstalling scipy-1.15.3:
      Successfully uninstalled scipy-1.15.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tsfresh 0.21.0 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.13.1 which is incompatible.


In [ ]:
pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 2.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-2.13.6-py3-none-any.whl.metadata (9.5 kB)
Using cached pybind11-2.13.6-py3-none-any.whl (243 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp311-cp311-linux_x86_64.whl size=4313509 sha256=a1a360735e1a6283bd9f4fdccf64d90e6c055d7647db6810e33b4894b87ffcc7
  Stored in directory: /root/.cache/pip/wheels/65/4f/35/5057db0249224e9ab55a513fa6b79451473ceb7713017823c3
Successfully built fasttext


In [ ]:
import pandas as pd
import numpy as np
from keras.models import Sequential
from keras.layers import Embedding, SimpleRNN, Dense
import fasttext
from keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN
from tensorflow.keras.layers import BatchNormalization
from keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
import tensorflow as tf
from transformers import create_optimizer
from transformers import TFAutoModelForSequenceClassification
from transformers import AutoTokenizer
from sklearn.metrics import classification_report

In [ ]:
!pip install tensorflow

In [ ]:
!pip install transformers

# **importing the preprocessed data**

In [ ]:
#the data is already preprocessed in the previous notebook so the only thing i did was downloading it and importing it here
df = pd.read_csv("/content/ready_data.csv")

In [ ]:
df.shape

(25000, 2)

In [ ]:
df.head(100)

,x,y
0,اراد تحدث طبيب,91
1,سمنه,25
2,اراد نظام رجيم مناسب,25
3,صداع غثيان,23
4,الم شديد جد جنب ايسر ظهر ٥ يوم مايفك ولاثانيه ...,14
...,...,...
95,سلام,23
96,تساقط شعر,25
97,Mild spondylodegenerative changes of the dorsa...,14
98,سمنه مفرط,25


In [ ]:
#splitting the text because the model only accepts it splited
df['x'] = df['x'].astype(str).apply(lambda x: x.split())

In [ ]:
#encoding the target
le = LabelEncoder()
y_encoded = le.fit_transform(df['y'])

# ***Neural network classification models***

# **preparing for RNN and LSTM**

In [ ]:
#downloading the pretrained word2vec(cbow) model
!wget https://archive.org/download/full_grams_cbow_300_twitter/full_uni_cbow_100_twitter.zip
!unzip full_uni_cbow_100_twitter.zip

--2025-06-07 05:45:22--  https://archive.org/download/full_grams_cbow_300_twitter/full_uni_cbow_100_twitter.zip
Resolving archive.org (archive.org)... 207.241.224.2
Connecting to archive.org (archive.org)|207.241.224.2|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://ia802803.us.archive.org/15/items/full_grams_cbow_300_twitter/full_uni_cbow_100_twitter.zip [following]
--2025-06-07 05:45:23--  https://ia802803.us.archive.org/15/items/full_grams_cbow_300_twitter/full_uni_cbow_100_twitter.zip
Resolving ia802803.us.archive.org (ia802803.us.archive.org)... 207.241.232.113
Connecting to ia802803.us.archive.org (ia802803.us.archive.org)|207.241.232.113|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 962251835 (918M) [application/zip]
Saving to: ‘full_uni_cbow_100_twitter.zip’

full_uni_cbow_100_t 100%[===================>] 917.67M  22.5MB/s    in 40s     

2025-06-07 05:46:03 (23.2 MB/s) - ‘full_uni_cbow_100_twitter.zip’ saved [9

In [ ]:
from gensim.models import Word2Vec
#loading the word2vec model using gensim and diving it the model we downloaded
word2vec_model = Word2Vec.load("full_uni_cbow_100_twitter.mdl")

In [ ]:
#downloading the pretrained arabic FastText model
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ar.300.bin.gz

--2025-06-07 05:46:38--  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ar.300.bin.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 3.169.55.29, 3.169.55.97, 3.169.55.43, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|3.169.55.29|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4500982519 (4.2G) [application/octet-stream]
Saving to: ‘cc.ar.300.bin.gz’

cc.ar.300.bin.gz    100%[===================>]   4.19G  93.2MB/s    in 47s     

2025-06-07 05:47:26 (90.9 MB/s) - ‘cc.ar.300.bin.gz’ saved [4500982519/4500982519]



In [ ]:
#unzipping the pretrained arabic FastText model
!gunzip cc.ar.300.bin.gz

In [ ]:
#loading the FastText model "cc.ar.300.bin.gz"
Fasttext_model = fasttext.load_model('cc.ar.300.bin')

In [ ]:
#sequance embedding using fasttext where for each word a vector of (300) is returned
FastText_seq = []
for s in df['x']:
   FastText_seq.append(np.array([Fasttext_model.get_word_vector(word) for word in s]))

In [ ]:
#sequance embedding using word2vec where for each word a vector of (100) is returned
word2vec_seq = []
for s in df['x']:
  word2vec_seq.append(np.array([word2vec_model.wv[word] for word in s if word in word2vec_model.wv]))

In [ ]:
#padding the embeddings of word2vec and fasttext so they all have the same length where some sentances have less words that other
FastText_padded = pad_sequences(FastText_seq, padding='post', dtype='float32')
Word2Vec_padded = pad_sequences(word2vec_seq, padding='post', dtype='float32')

In [ ]:
#making sue that y and x have the same length
y = df['y'].values[:len(FastText_padded)]

In [ ]:
#splitting the prepared embeddings and encoded y
X_train_ft, X_test_ft, y_train_ft, y_test_ft = train_test_split(FastText_padded,y_encoded, test_size=0.2, random_state=42)
X_train_w2v, X_test_w2v, y_train_w2v, y_test_w2v = train_test_split(Word2Vec_padded, y_encoded, test_size=0.2, random_state=42)

In [ ]:
classes_num = 5

# **RNN with FastText**

In [ ]:
# initializing the model
model_RNN_FT = Sequential()
#adding a layer with SimpleRNN model
model_RNN_FT.add(SimpleRNN(64,input_shape=(X_train_ft.shape[1], X_train_ft.shape[2])))
#Add Dense layer with relu and softmax activation functions (softmax because we have multiclass classification)
model_RNN_FT.add(Dense(64, activation='relu'))
model_RNN_FT.add(Dense(classes_num, activation='softmax'))
# Compile and training the model using adam optimizer and sparse_categorical_crossentropy loss because our classification problem nature
model_RNN_FT.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_RNN_FT.summary()
#fitting the RNN model using fasttext training data
model_RNN_FT.fit(X_train_ft, y_train_ft, epochs=10, batch_size=16)
# Evaluating the model
loss, accuracy = model_RNN_FT.evaluate(X_test_ft, y_test_ft)
print(f"Test Accuracy: {accuracy:.4f}")


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 64)             │        23,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,845 (108.77 KB)

 Trainable params: 27,845 (108.77 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.2538 - loss: 1.6025
Epoch 2/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.2683 - loss: 1.5924
Epoch 3/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.2700 - loss: 1.5934
Epoch 4/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.2791 - loss: 1.5785
Epoch 5/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.3227 - loss: 1.5160
Epoch 6/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.3241 - loss: 1.5098
Epoch 7/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.3240 - loss: 1.5097
Epoch 8/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.3218 - loss: 1.5102
Epoch 9/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.3285 - loss: 1.5115
Epoch 10/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.3199 - loss: 1.5122
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.3180 - loss: 1.5231
Test Accuracy: 0.3226


# **RNN with Word2vec**

In [ ]:
# initializing the model
model_RNN_W2V = Sequential()
#adding a layer with SimpleRNN model
model_RNN_W2V.add(SimpleRNN(64,input_shape=(X_train_w2v.shape[1], X_train_w2v.shape[2])))
#Add Dense layer with relu and softmax activation functions (softmax because we have multiclass classification)
model_RNN_W2V.add(Dense(64, activation='relu'))
model_RNN_W2V.add(Dense(classes_num, activation='softmax'))
# Compile and training the model using adam optimizer and sparse_categorical_crossentropy loss because our classification problem nature
model_RNN_W2V.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_RNN_W2V.summary()
#fitting the RNN model using fasttext training data
model_RNN_W2V.fit(X_train_w2v, y_train_w2v, epochs=10, batch_size=16)
# Evaluate
loss, accuracy = model_RNN_W2V.evaluate(X_test_w2v, y_test_w2v)
print(f"Test Accuracy: {accuracy:.4f}")


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_1 (SimpleRNN)        │ (None, 64)             │        10,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,045 (58.77 KB)

 Trainable params: 15,045 (58.77 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.2582 - loss: 1.6010
Epoch 2/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.2738 - loss: 1.5924
Epoch 3/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.2720 - loss: 1.5906
Epoch 4/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.2698 - loss: 1.5964
Epoch 5/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.2692 - loss: 1.5922
Epoch 6/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.2716 - loss: 1.5912
Epoch 7/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.2639 - loss: 1.5990
Epoch 8/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.2702 - loss: 1.5927
Epoch 9/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.2701 - loss: 1.5912
Epoch 10/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.2766 - loss: 1.5899
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.2800 - loss: 1.5874
Test Accuracy: 0.2714


#**FastText emmbeding worked better than Word2vec that is why i will use it with LSTM**

In [ ]:
# initializing the model
model_LSTM = Sequential()
#adding a layer with LSTM model
model_LSTM.add(LSTM(64, input_shape=(X_train_ft.shape[1], X_train_ft.shape[2])))
#Add Dense layer with relu and softmax activation functions (softmax because we have multiclass classification)
model_LSTM.add(Dense(64, activation='relu'))
model_LSTM.add(Dense(classes_num, activation='softmax'))
# Compile and training the model using adam optimizer and sparse_categorical_crossentropy loss because our classification problem nature
model_LSTM.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_LSTM.summary()
#fitting the LSTM model using fasttext training data
model_LSTM.fit(X_train_ft, y_train_ft, epochs=10, batch_size=16)
# Evaluating the model
loss, accuracy = model_LSTM.evaluate(X_test_ft, y_test_ft)
print(f"Test Accuracy: {accuracy:.4f}")

/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        93,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 97,925 (382.52 KB)

 Trainable params: 97,925 (382.52 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.2680 - loss: 1.5951
Epoch 2/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.2723 - loss: 1.5906
Epoch 3/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.2716 - loss: 1.5901
Epoch 4/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.2693 - loss: 1.5916
Epoch 5/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.2723 - loss: 1.5891
Epoch 6/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.2709 - loss: 1.5904
Epoch 7/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.2688 - loss: 1.5911
Epoch 8/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.2719 - loss: 1.5903
Epoch 9/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.2703 - loss: 1.5910
Epoch 10/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.2716 - loss: 1.5899
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.2800 - loss: 1.5873
Test Accuracy: 0.2714


# ***Transformers***

# **loading the original data because transformers deals with raw data**

In [ ]:
df_trans = pd.read_csv("/content/altibbi_specialty_data.csv")

In [ ]:
#the data is big so we took 25000 random samples of it
df_trans = df_trans.drop("name_ar", axis=1)
df_trans = df_trans.sample(n=25000, random_state=42)

# **BERT transformer**

**preparing the data for bert**

In [ ]:
#initializing arabbert Auto toknizer using("aubmindlab/bert-base-arabertv02")
bert_toknizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv02")
#encoding the target
df_trans['specialty_id'] = le.fit_transform(df_trans['specialty_id'])
#here we are transforming the data to a form that bert understands where we use padding to ensure that all the inputs have the same length
#ensuring that all texts are shorter than 130 tokens
#and lastly ensuring that the output works with tensorflow
encodings_tokenization_bert = bert_toknizer(list(df_trans["question_body"].astype(str)),padding=True,truncation=True,max_length=130,return_tensors='tf')
#attention mask that helps the model now the difference between padding tokens and actual tokens
attention_mask_bert = encodings_tokenization_bert['attention_mask'].numpy()
#input formed as toknized numpy array
input_bert = encodings_tokenization_bert['input_ids'].numpy()
#the target as numpy array
target = tf.convert_to_tensor(df_trans['specialty_id']).numpy()
#splitting and preparing testing and training data
#%80 training rest testing
X_train_in_bert, X_test_in_bert, X_train_attention_bert, X_test_attention_bert, y_train_bert, y_test_bert = train_test_split(input_bert, attention_mask_bert, target, test_size=0.2, random_state=42)
# Train and test datasets using tensor slices where batch = 8
train_bert = tf.data.Dataset.from_tensor_slices(({'input_ids': X_train_in_bert,'attention_mask': X_train_attention_bert}, y_train_bert)).batch(8)
test_bert = tf.data.Dataset.from_tensor_slices(({'input_ids': X_test_in_bert,'attention_mask': X_test_attention_bert}, y_test_bert)).batch(8)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/825k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
#Initial learning rate = init_lr=3e-5
#	num_train_steps = number of batches(8) × epochs(4)
optimizer_bert, schedule_bert = create_optimizer(init_lr=3e-5,num_warmup_steps=int(0.1*len(train_bert) * 4),num_train_steps=len(train_bert) * 4)

In [ ]:
#preparing the arabertmodel
#TFAutoModelForSequenceClassification is a tenserflow version of bert that is spicifcally for classification tasks
bert_model = TFAutoModelForSequenceClassification.from_pretrained("aubmindlab/bert-base-arabertv02", num_labels=5)

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertForSequenceClassification: ['bert.embeddings.position_ids']
- This IS expected if you are initializing TFBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
#compiling the bert model using the optimizer we initialized and setting the loss function to SparseCategoricalCrossentropy which is user for classification task
bert_model.compile(optimizer=optimizer_bert,loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),metrics=['accuracy'])

In [ ]:
#finaly fitting the model
bert_model.fit(train_bert, validation_data=test_bert, epochs=4)

Epoch 1/4
2500/2500 [==============================] - 694s 262ms/step - loss: 0.4881 - accuracy: 0.8282 - val_loss: 0.3090 - val_accuracy: 0.8998
Epoch 2/4
2500/2500 [==============================] - 649s 260ms/step - loss: 0.2638 - accuracy: 0.9104 - val_loss: 0.3163 - val_accuracy: 0.9032
Epoch 3/4
2500/2500 [==============================] - 648s 259ms/step - loss: 0.1977 - accuracy: 0.9334 - val_loss: 0.3158 - val_accuracy: 0.9030
Epoch 4/4
2500/2500 [==============================] - 648s 259ms/step - loss: 0.1510 - accuracy: 0.9487 - val_loss: 0.3365 - val_accuracy: 0.9044


In [ ]:
#bert predictions
bert_predictions = bert_model.predict(test_bert)
#logits are raw scores before applying softmax, where each with its corusponding class and sample
logits = bert_predictions.logits
#this returns the maximum logit which shows which class is the predicted
max_class = tf.argmax(logits, axis=1).numpy()
actual_y = []
for b in test_bert:
    _, labels = b
    np.array(actual_y.extend(labels.numpy()))

625/625 [==============================] - 55s 84ms/step


In [ ]:
print(classification_report(actual_y, max_class))

              precision    recall  f1-score   support

           0       0.91      0.92      0.91      1357
           1       0.95      0.93      0.94      1020
           2       0.92      0.93      0.92       810
           3       0.90      0.87      0.89       782
           4       0.85      0.87      0.86      1031

    accuracy                           0.90      5000
   macro avg       0.91      0.90      0.90      5000
weighted avg       0.90      0.90      0.90      5000

